# asyncio 科学学习笔记本

> 目标：在**真实项目**中自如使用 asyncio，而不是只会看 demo。
> 学习原则：先跑通，再理解；先官方，后二手；先常用，后边缘；先项目，后原理；先输出，后遗忘。

本笔记本结合 `LangChain_LawAgent` 项目本身取材——法条文件、案例库、嵌入模型这些
真实对象就是练习素材，不用另造玩具数据。

## 十阶段路径与进度

| 阶段 | 内容 | 本子状态 |
|------|------|----------|
| 0 | 诊断基础与目标 | 待你填写 |
| 1 | 定目标与验收标准 | 已给模板 |
| 2 | 建立知识地图 | 已给 |
| 3 | 提取最小必要知识 | 已给 |
| 4 | 搭环境，跑通最小闭环 | **可运行** |
| 5 | 建立核心心智模型 | **可运行** |
| 6 | 刻意练习 | 任务书已给，答案待你做完再看 |
| 7 | 项目驱动 | 未解锁 |
| 8 | 工程化与排错 | 未解锁 |
| 9 | 源码与原理 | 未解锁 |
| 10 | 输出、复习、迁移 | 未解锁 |

## 本子约定

- Jupyter 单元格**本身已运行在事件循环里**，所以直接写 `await main()`。
- 写成 `.py` 脚本时，入口才用 `asyncio.run(main())`。两者不可互换，原因见阶段 5 破坏实验。
- 运行本子用 `lawapp` 内核（对应 `F:\Anaconda_env\lawApp_langGraph\python.exe`）。


## 阶段 0 · 诊断基础与目标

先别往下写代码。回答下面四问——答案直接写在这个单元格下面，或另开 markdown 单元格。

1. **当前基础**：Python 基础语法熟练吗？写过 `async def` 吗？写过装饰器/生成器吗？
2. **目标场景**：学 asyncio 是为了什么？（本项目：FastAPI 端点 + LangGraph 图执行 + MCP stdio + RAG 检索）
3. **Python 版本**：本环境是 **3.11.15**，所以 `asyncio.TaskGroup`、`asyncio.timeout()` 都可用。
4. **时间投入**：每天多久？期望多久达到目标？

### 我的回答

<!-- 把你的答案写在这里 -->

### 为什么要先诊断

asyncio 的坑分三层：**语法层**（漏 await）、**调度层**（阻塞事件循环）、
**工程层**（超时/取消/测试）。基础不同，切入层不同。跳过诊断直接抄代码，
通常会稳定卡在调度层——代码能跑，但并发是假的。

> 填完回答后说「继续」，进入阶段 1。


## 阶段 1 · 定目标与验收标准

目标写得含糊，验收就没法做。把「学会 asyncio」翻译成可检查的行为。

### 本项目的验收标准（对照你的实际需求改写）

| # | 我能做到 | 怎么验证 |
|---|----------|----------|
| 1 | 解释协程 / 事件循环 / Task / await 四者关系 | 不看笔记讲 90 秒，讲给橡皮鸭也行 |
| 2 | 写并发程序并控制并发数 | 用 `Semaphore` 限制同时进行的任务数，打印活跃数不超标 |
| 3 | 排查「阻塞事件循环」 | 看心跳打点是否均匀；用 `debug=True` 拿到警告 |
| 4 | 排查「协程未等待」 | 复现 `coroutine ... was never awaited` 警告并修掉 |
| 5 | 正确处理超时与取消 | `asyncio.timeout()` 包裹，`CancelledError` 正确向上传播 |
| 6 | 把同步 I/O 改造成异步 | 把某个 `requests.get` / 同步读库改成 `to_thread` 或异步客户端 |
| 7 | 写异步测试 | 用 `pytest-asyncio` 跑通一个 `async def test_*` |

### 我的目标

<!-- 按上表格式写你自己的 5-7 条，带验证方式 -->


## 阶段 2 · 知识地图

不要一次学完。按圈层推进，每圈都能独立用起来。

```
                        进阶圈 (需要时再看)
        ┌─────────────────────────────────────────────┐
        │  事件循环底层 API · Future · 自定义策略       │
        │  uvloop · Task 内幕 · 调试模式               │
        │                                             │
        │        应用圈 (工程必须)                     │
        │   ┌───────────────────────────────────┐     │
        │   │  Semaphore  Queue  timeout()       │     │
        │   │  TaskGroup  to_thread  取消与清理   │     │
        │   │                                   │     │
        │   │        核心圈 (每天用)             │     │
        │   │   ┌─────────────────────────┐     │     │
        │   │   │  async def / await      │     │     │
        │   │   │  asyncio.run()          │     │     │
        │   │   │  create_task()          │     │     │
        │   │   │  gather() / sleep()     │     │     │
        │   │   └─────────────────────────┘     │     │
        │   └───────────────────────────────────┘     │
        └─────────────────────────────────────────────┘
```

### 映射到本项目

| 圈层 | 项目里的对应物 | 位置 |
|------|----------------|------|
| 核心 | FastAPI 端点内的 `await`、图执行 | `lawApp_LangGraph/FastAPI/api.py` |
| 核心 | SSE 流的 `async for` | `api.py` 的 `/ask/stream` |
| 应用 | `asyncio.to_thread` 包同步重活 | `RAG_service/embedder.py:66`、`tools/tools.py:48` |
| 应用 | MCP 子进程 stdio 会话 | `lawApp_LangGraph/mcp_client.py` |
| 应用 | 图节点的并发工具调用 | `lawApp_LangGraph/LangGraph_lawApp.py` |
| 进阶 | LangGraph 的 checkpointer / store 异步后端 | `lawApp_LangGraph/runtime.py` |

> 记住这张表。你在本子里学的每个概念，最后都要能指回项目里的一行代码。


## 阶段 3 · 最小必要知识

只讲够用的。四个概念，一个流程图。

### 1. 协程函数 vs 协程对象

`async def f()` 定义的是**协程函数**。调用它 `f()` **不会执行函数体**，
只返回一个**协程对象**——一张「待办单」，不是「已完成」。

> 比喻：协程对象是写好的菜谱，不是端上来的菜。

### 2. 事件循环

一个单线程的调度器。它手里维护一张待办清单，反复做一件事：
**挑一个能推进的任务，推进到它主动让出为止，再挑下一个。**

单线程意味着：任何一段不主动让出的同步代码，都会**卡住所有人**。

### 3. Task

`asyncio.create_task(coro)` 把协程对象**登记进事件循环的待办清单**，返回 `Task`。
登记之后，你不 `await` 它也会被调度——这是「并发」和「顺序」的分水岭。

`asyncio.gather(...)` 内部就是帮你批量做这件事，再等全部完成。

### 4. await

`await` 做两件事：

1. 把当前协程**挂起**，控制权交还事件循环；
2. 等目标完成后再**恢复**当前协程，并取出结果。

所以 `await` 的意思是「我在这儿等，但**别人可以接着跑**」，
而不是 `time.sleep` 那种「所有人都停下来等我」。

### 流程图：一次 gather 的执行

```
你:      await asyncio.gather(A(), B())
              │
   事件循环:  ├─ 登记 A → Task
              ├─ 登记 B → Task
              │
              ├─ 跑 Task A ──── A 遇到 await asyncio.sleep ──┐ 让出
              │                                              │
              ├─ 跑 Task B ──── B 遇到 await asyncio.sleep ──┤ 让出
              │                                              │
              ├─ 都在等待, 循环空转 (或去跑别的任务)          │
              │                                              │
              ├─ A 的等待到期 ←───────────────────────────────┘
              ├─ 跑 Task A ──── A 返回结果
              ├─ B 的等待到期
              ├─ 跑 Task B ──── B 返回结果
              │
你:      拿到 [A 的结果, B 的结果]
```

关键：**A 和 B 的等待是重叠的**。这就是并发的全部收益来源。

### 什么时候并发有用

收益来自**等待**，不来自**计算**。

| 场景 | 并发有用吗 |
|------|-----------|
| 网络请求（数据库 / HTTP / 模型推理） | 有用，收益巨大 |
| 等待外部进程 / 子进程 | 有用 |
| 本地磁盘小文件读取 | 基本没用，甚至更慢 |
| CPU 密集计算 | 没用，要用多进程 |

> 这一点在阶段 4 的最后一段代码里会亲手验证。


## 阶段 4 · 搭环境，跑通最小闭环

先跑通，再理解。先做环境自检，再跑两段做**同一件事**的代码：
取 5 条法条，每条耗时 0.5 秒。第一段同步串行，第二段异步并发。
先跑，看数字，再回头读解释。

### 环境自检


In [ ]:
import asyncio
import sys
import time
from importlib.metadata import PackageNotFoundError, version

print(f"Python: {sys.version.split()[0]}")

# 第一个真实的 asyncio 坑: 没有运行中的事件循环时, get_running_loop() 会抛异常,
# 而不是返回 None。所以判断"当前是否在循环里"必须 try/except。
try:
    asyncio.get_running_loop()
    in_loop = True
except RuntimeError:
    in_loop = False
print(f"当前是否已在事件循环中: {in_loop}")

print()
# 本项目异步栈的关键包(本子的核心示例不依赖它们, 仅用于确认项目环境可用)
# 用 importlib.metadata 取版本而非模块的 __version__: 部分包(如 langgraph)不暴露该属性
# 刻意不查 sentence_transformers: 冷导入要 20 秒以上, 会拖慢本子每次运行
for dist in ("langgraph", "langchain-core", "httpx"):
    try:
        print(f"  {dist:<24} {version(dist)}")
    except PackageNotFoundError:
        print(f"  {dist:<24} 未安装")

print()
print("本子用裸 await；.py 脚本用 asyncio.run(main())")


### 4.1 同步版


In [ ]:
def fetch_article(article_id: int) -> str:
    """模拟「取一条法条」的 I/O。

    真实项目里这一步是读文件、查 pgvector、或发 HTTP —— 全都是「等」。
    这里用 time.sleep 假装在等。
    """
    time.sleep(0.5)
    return f"《民法典》第{article_id}条"


def main_sync() -> list[str]:
    t0 = time.perf_counter()
    articles = [fetch_article(i) for i in range(5)]
    print(f"[同步串行] 5 条法条耗时 {time.perf_counter() - t0:.2f}s")
    return articles


articles = main_sync()
print(articles)


### 4.2 异步版


In [ ]:
async def fetch_article_async(article_id: int) -> str:
    """与 fetch_article 做同一件事, 但等待方式可让出。"""
    await asyncio.sleep(0.5)          # 非阻塞等待: 挂起自己, 让出控制权
    return f"《民法典》第{article_id}条"


async def main_async() -> list[str]:
    t0 = time.perf_counter()
    # gather 把每个协程包成 Task 并发调度, 全部完成后按原顺序返回结果列表
    articles = await asyncio.gather(*(fetch_article_async(i) for i in range(5)))
    print(f"[异步并发] 5 条法条耗时 {time.perf_counter() - t0:.2f}s")
    return articles


articles = await main_async()          # Jupyter 里直接 await; 脚本里写 asyncio.run(main_async())
print(articles)


### 4.3 逐行解释

**同步版**

| 行 | 说明 |
|----|------|
| `time.sleep(0.5)` | 阻塞式等待。**整个线程**停 0.5 秒，谁都动不了 |
| `[fetch_article(i) for i in range(5)]` | 列表推导，一个接一个。总耗时 = 5 × 0.5 = 2.5s |

**异步版**

| 行 | 说明 |
|----|------|
| `async def fetch_article_async(...)` | 协程函数。调用它只得到协程对象，不执行函数体 |
| `await asyncio.sleep(0.5)` | 非阻塞等待。挂起当前协程，把控制权交还事件循环 |
| `*(... for i in range(5))` | 生成器解包，等价于 `gather(coro0, coro1, ..., coro4)` |
| `await asyncio.gather(...)` | 批量登记为 Task 并等待全部完成，结果**按传入顺序**返回 |
| `= await main_async()` | 顶层 await。Jupyter 支持；普通脚本要写 `asyncio.run(main_async())` |

**为什么是 ≈0.5s 而不是 ≈2.5s**

5 个协程的等待区间完全重叠：全部登记后，事件循环在 0.5 秒处同时唤醒 5 个任务。
（实跑是 2.50s → 0.52s，多出的 0.02s 是调度开销。）

**关键前提**：`fetch_article_async` 里用的是 `asyncio.sleep`，它会让出。
如果换成 `time.sleep`，耗时立刻退回 2.5s —— 见阶段 5 的破坏实验。


### 4.4 回到本项目：并发的收益不是处处都有

上面用 `time.sleep` 假装等待，所以差距悬殊。换成**真实但极快**的本地操作，
并发可能一点忙都帮不上。亲手验证一下。


In [ ]:
from pathlib import Path

# 与仓库其它 notebook 一致: 从 cwd 向上找到仓库根
ROOT = next(p for p in (Path.cwd(), Path.cwd().parent)
            if (p / "lawApp_LangGraph").is_dir())
LAW_DIR = ROOT / "data" / "Documents" / "LawDocument"


def read_law_sync(name: str) -> int:
    """同步读一个法条文件, 返回字符数。"""
    return len((LAW_DIR / name).read_text(encoding="utf-8"))


async def read_law_via_thread(name: str) -> int:
    """本项目真实做法(见 RAG_service/embedder.py:66):

    同步重活丢进线程池, 事件循环保持可调度。
    """
    return await asyncio.to_thread(read_law_sync, name)


names = sorted(p.name for p in LAW_DIR.glob("*.txt"))
print(f"待读 {len(names)} 个法条文件")

t0 = time.perf_counter()
sync_sizes = [read_law_sync(n) for n in names]
print(f"[同步串行]   {time.perf_counter() - t0:.4f}s  共 {sum(sync_sizes)} 字")

t0 = time.perf_counter()
async_sizes = await asyncio.gather(*(read_law_via_thread(n) for n in names))
print(f"[线程池并发] {time.perf_counter() - t0:.4f}s  共 {sum(async_sizes)} 字")


### 读这段结果

实跑输出（你的机器上数字会不同，量级应当接近）：

```
待读 7 个法条文件
[同步串行]   0.0103s  共 142720 字
[线程池并发] 0.0049s  共 142720 字
```

两个数字**都不到 10 毫秒**。这里异步版快了一点，但请不要据此下结论——
在这个量级上，线程池调度开销和测量噪声是同一数量级，快慢基本由运气决定。
多跑几次，两个数字会在几毫秒内互相反超。

**真正的结论是这个**：

- 并发收益来自**回填等待时间**。本地读 7 个小文件几乎不等待，
  所以没有等待可以回填——加速空间本来就只有那 10 毫秒。
- `asyncio.to_thread` 的主要价值不是「让这一次操作变快」，
  而是**不让事件循环被卡住**。收益要等有别的任务在跑时才体现出来。
- 把 `read_text` 换成一次数据库查询或一次嵌入模型推理
  （本项目里动辄几百毫秒），差距立刻出现。

> 一句话：**先确认瓶颈是等待，再上并发。** 本地小文件不是等得起的对象。

本项目里真正值得并发的位置就在这里 —— 嵌入、重排、PDF 渲染、SerpAPI 查询，
全都是「同步库 + 长时间等待」，所以项目统一用 `asyncio.to_thread` 包起来。

**动手验证**：给这个实验加一个心跳 ticker（照实验 3 的写法），再让
`read_law_sync` 里加一句 `time.sleep(0.5)` 模拟慢 I/O。对比 `to_thread` 版和
直接同步调用版的打点分布——你会看到实验 3 和实验 4 的差别在真实代码里重现。


## 阶段 5 · 建立核心心智模型

四个实验。每个都先跑，观察输出，再看结论。
**不要跳过破坏实验** —— 你对 asyncio 的理解，主要来自看它怎么坏。


### 实验 1 · 顺序 await 和 gather 差在哪

两者代码只差一层括号，耗时差 5 倍。


In [ ]:
async def main_sequential() -> list[str]:
    t0 = time.perf_counter()
    # 顺序 await: 每个都等完了才登记下一个 —— 等于串行
    articles = [await fetch_article_async(i) for i in range(5)]
    print(f"[顺序 await] 耗时 {time.perf_counter() - t0:.2f}s")
    return articles


_ = await main_sequential()
print("对比上面 gather 版的 ≈0.5s —— 差别只在「有没有同时登记」")


**心智模型**：`await` 本身**不产生并发**。它只是「等」。
并发来自「先把多个任务登记进循环」，也就是 `create_task` / `gather` / `TaskGroup`。

一句话记住：**先登记，再等待。**


### 实验 2 · 破坏：漏写 await

最常见的错误。不报错，不崩溃，只是拿到一个不该存在的东西。


In [ ]:
# 错误写法: 调用了协程函数, 但没有 await
pending = fetch_article_async(1)

print("拿到的类型:", type(pending).__name__)
print("拿到的值  :", pending)
print("→ 这不是结果, 是一张还没下锅的菜谱")

pending.close()   # 关掉它, 避免 GC 时刷出 "coroutine was never awaited" 警告

print()
# 正确写法
result = await fetch_article_async(1)
print("await 之后:", result)


**这条错误的三种表现**（都见过才算过关）：

| 表现 | 场景 |
|------|------|
| `RuntimeWarning: coroutine '...' was never awaited` | 在协程内部漏写，GC 时警告 |
| 拿到 `<coroutine object ...>` 而不是数据 | 直接使用返回值 |
| **完全无声** | 漏掉的是「用于产生副作用」的调用，比如漏 await 一个写库操作 |

第三种最危险。所以项目里凡是「只调不用结果」的异步函数，都要在 review 里重点看。


### 实验 3 · 破坏：用 time.sleep 阻塞事件循环

这是本项目最需要你建立的直觉。真实症状是「服务偶发卡顿」「加了并发反而更慢」。

下面用心跳任务（ticker）观察事件循环有没有被占住。**先看输出再读结论。**


In [ ]:
async def ticker(name: str, ticks: int, interval: float, t0: float) -> None:
    """心跳任务: 定期打点。用来观察事件循环是否还在调度别人。"""
    for i in range(ticks):
        await asyncio.sleep(interval)
        print(f"    [{name}] 第 {i + 1} 次打点 (t={time.perf_counter() - t0:.2f}s)")


async def blocking_wait() -> None:
    """元凶: 同步 sleep。整整 1 秒里事件循环完全停摆。"""
    time.sleep(1.0)


t0 = time.perf_counter()
await asyncio.gather(ticker("A", 4, 0.25, t0), blocking_wait())
print(f"    总耗时 {time.perf_counter() - t0:.2f}s")


**看输出**（实跑结果）：

```
    [A] 第 1 次打点 (t=1.00s)
    [A] 第 2 次打点 (t=1.26s)
    [A] 第 3 次打点 (t=1.52s)
    [A] 第 4 次打点 (t=1.79s)
    总耗时 1.79s
```

`A` 的 4 次打点**全部挤在 1.0s 之后**，而不是均匀分布。
这就是事件循环被 `time.sleep` 占死的现场——ticker 早就该醒了，但没人能调度它。

**总耗时 1.79s 是怎么来的**，这个推演值得走一遍：

| 时刻 | 发生了什么 |
|------|-----------|
| 0.00s | 两个任务登记。ticker 的定时器设在 0.25s，`blocking_wait` 开始阻塞 |
| 0.25s | ticker 的定时器**到期了**，但线程被 `time.sleep` 占着，没人能来收 |
| 1.00s | 阻塞结束，事件循环重获控制权。它这才发现 ticker 的定时器早已过期 |
| 1.00s | 第 1 次打点（迟到 0.75 秒） |
| 1.00 → 1.79s | ticker 剩下的 3 次 0.25s 等待才依次走完 |

对比实验 4 的 `B`：打点均匀铺在 0.25 / 0.51 / 0.78 / 1.04 秒，总耗时 **1.04s**。

**所以阻塞版的代价是双重的**：不只是慢（1.79s vs 1.04s），更是**失去响应性**——
期间任何定时器、任何其他请求都得不到服务。

**它为什么危险**：在 FastAPI 里，一个同步 `requests.get` 会卡住**所有**并发请求。
你加了并发、开了多任务，结果吞吐反而掉了。

**这条规则要刻进肌肉记忆**：

| 绝对不要（在协程里） | 改成 |
|----------------------|------|
| `time.sleep()` | `await asyncio.sleep()` |
| `requests.get()` | `await httpx.AsyncClient().get()` |
| 同步数据库驱动 | 异步驱动，或 `await asyncio.to_thread(...)` |
| 大文件同步读 | `await asyncio.to_thread(...)` |
| 同步模型推理（本项目！） | `await asyncio.to_thread(...)` |

最后一行的原型就在 `RAG_service/embedder.py:66`：

```python
return await asyncio.to_thread(embed_query_sync, text)
```


### 实验 4 · 修复：换成可让出的等待

同一段代码，只把 `time.sleep(1.0)` 换成 `await asyncio.sleep(1.0)`。


In [ ]:
async def cooperative_wait() -> None:
    """可让出的等待。"""
    await asyncio.sleep(1.0)


t0 = time.perf_counter()
await asyncio.gather(ticker("B", 4, 0.25, t0), cooperative_wait())
print(f"    总耗时 {time.perf_counter() - t0:.2f}s")


**看输出**（实跑结果）：

```
    [B] 第 1 次打点 (t=0.25s)
    [B] 第 2 次打点 (t=0.51s)
    [B] 第 3 次打点 (t=0.78s)
    [B] 第 4 次打点 (t=1.04s)
    总耗时 1.04s
```

`B` 的 4 次打点**均匀铺开**，每一次都在它该醒的时刻醒来。

把两次实验并排看：

| | 打点时刻 | 总耗时 |
|---|---|---|
| 实验 3（`time.sleep`） | 1.00 / 1.26 / 1.52 / 1.79 —— 挤在末尾 | 1.79s |
| 实验 4（`await asyncio.sleep`） | 0.25 / 0.5 / 0.8 / 1.0 —— 均匀铺开 | 1.0s |

（每次运行有几十毫秒抖动；要紧的是**分布形状**，不是小数点后两位。）

**这就是「阻塞」和「非阻塞」的可观测差别。**
排错时，这两个实验就是你的诊断工具：往可疑代码旁边挂一个 ticker，
打点一挤，凶手就找到了。


### 阶段 5 小结

**常见坑**

1. 漏 `await` —— 三种表现，见实验 2。
2. 协程里写同步阻塞调用 —— 见实验 3。判断方法：这个调用会不会让线程等？
3. 以为 `await` 就等于并发 —— 见实验 1。先登记，再等待。
4. 在 `async def` 里调用 `asyncio.run()` —— 会抛
   `RuntimeError: asyncio.run() cannot be called from a running event loop`。
   Jupyter / FastAPI / 已有循环的环境里，直接 `await` 即可。
5. 把 CPU 密集任务塞进 asyncio —— 换进程池，或 `to_thread` 后接受 GIL 限制。

**验收标准**（做到才算过关）

- [ ] 能不看笔记讲清协程 / 事件循环 / Task / await 四者关系
- [ ] 能说出实验 1 两种写法耗时差几倍、为什么
- [ ] 能自己复现「漏 await」警告并修掉
- [ ] 能看着心跳打点判断事件循环有没有被阻塞
- [ ] 能指出本项目里至少 3 处 `asyncio.to_thread` 并说明为什么需要它

**自测问题**（先合上笔记回答，再回上去对）

1. `async def f(): ...` 之后写 `f()`，得到什么？函数体执行了没有？
2. `await` 到底做了什么？它和 `time.sleep` 的本质区别是什么？
3. `asyncio.gather(*coros)` 和逐个 `await coro` 的差别在哪一层？
4. 一段协程代码耗时 2.5 秒，改成 `gather` 后还是 2.5 秒。最可能的原因是什么？
5. 什么时候不该用 asyncio？举一个本项目的例子。

**复习节点**：1 天后 · 3 天后 · 1 周后 · 1 月后


## 阶段 6 · 刻意练习

**先自己写，写完再看答案。** 答案在下一个单元格，现在不要往下滚。

### 练习 A · 限流

把 `fetch_article_async` 并发跑 20 次，但**同时最多只允许 3 个在跑**。
每完成一个，打印当前已完成数量。

要求：用 `asyncio.Semaphore`。写完后运行，确认打印出的并发数没有超过 3。

### 练习 B · 超时

写一个 `slow_article(delay)` 协程，`delay` 从 0.1 到 1.0。
用 `asyncio.timeout()` 给整批任务设 0.5 秒上限。
超时后要能打印「哪些完成了、哪些被取消了」，并且**不抛异常到单元格外面**。

### 练习 C · 改造本项目代码

打开 `lawApp_LangGraph/tools/tools.py`，找到第 48 行附近的
`raw = await asyncio.to_thread(search.results, query)`。

回答三个问题（写在下面）：

1. 如果去掉 `to_thread` 直接写 `search.results(query)`，会发生什么？用实验 3 的
   心跳打点法证明你的判断。
2. 为什么这里用 `to_thread` 而不是换成异步 HTTP 客户端？（提示：谁提供的 SDK）
3. 如果 `search.results` 抛异常，这个异常会传播到哪一层？会被谁 catch？


### 参考答案 · 练习 A

```python
async def limited_fetch(sem: asyncio.Semaphore, article_id: int) -> str:
    async with sem:                      # 进入时占用一个名额, 退出时释放
        await asyncio.sleep(0.5)
        return f"《民法典》第{article_id}条"


async def main_limited(total: int = 20, limit: int = 3) -> list[str]:
    sem = asyncio.Semaphore(limit)
    t0 = time.perf_counter()
    tasks = [asyncio.create_task(limited_fetch(sem, i)) for i in range(total)]
    done = 0
    for coro in asyncio.as_completed(tasks):
        await coro
        done += 1
        print(f"  完成 {done}/{total}  (t={time.perf_counter() - t0:.2f}s)")
    return [t.result() for t in tasks]
```

耗时约 `ceil(20 / 3) × 0.5 ≈ 3.5s`。若没有 Semaphore，只要 0.5s ——
**限流是用时间换稳定**，别误以为 Semaphore 能加速。

### 参考答案 · 练习 B

```python
async def slow_article(delay: float) -> str:
    await asyncio.sleep(delay)
    return f"delay={delay}"


async def main_timeout() -> None:
    tasks = [asyncio.create_task(slow_article(d / 10)) for d in range(1, 11)]
    try:
        async with asyncio.timeout(0.5):     # Python 3.11+
            await asyncio.gather(*tasks)
    except TimeoutError:
        done = [t.result() for t in tasks if t.done() and not t.cancelled()]
        pending = [t for t in tasks if not t.done()]
        print(f"超时。已完成 {len(done)} 个: {done}")
        print(f"未完成 {len(pending)} 个, 逐个取消")
        for t in pending:
            t.cancel()
        await asyncio.gather(*pending, return_exceptions=True)
```

关键点：`asyncio.timeout()` 到期后只抛 `TimeoutError`，**不会自动取消**你手动
`create_task` 出去的任务。必须自己清理，否则任务泄漏、警告刷屏。

Python 3.10 及以下没有 `asyncio.timeout()`，用 `asyncio.wait_for(coro, t)` 替代。

### 参考答案 · 练习 C

1. 去掉 `to_thread` 后，`search.results(query)` 是同步网络请求（SerpAPI），
   会在事件循环线程里阻塞到请求返回。用 ticker 打点法可以观测到：
   打点停止若干秒，然后一次性涌出。**在 FastAPI 里这等于卡住所有并发请求。**
2. `serpapi` 是同步 SDK，没有官方异步客户端，所以只能 `to_thread`。
   如果有异步客户端，优先换客户端 —— 线程池有开销，且受默认线程数限制。
3. 传播到 `tools.py` 里该工具函数的调用方，最终由 LangGraph 的 `ToolNode`
   捕获并写进图状态的错误字段。这也是为什么项目里工具函数普遍返回
   `{"status": "error", ...}` 而不是直接抛 —— 让 LLM 能看到失败并重规划。


## 下一步

阶段 0-5 已经跑通。填完阶段 0 的诊断和阶段 1 的目标，做完阶段 6 的三道练习，
就可以进阶段 7。

### 阶段 7 预告 · 项目驱动

按原定路径：

| 项目 | 技能点 | 本项目可用素材 |
|------|--------|----------------|
| 1. 并发图片下载器 | aiohttp、gather、Semaphore | 换成并发抓取案例 MD 里的链接 |
| 2. 异步端口扫描器 | `open_connection`、`wait_for`、异常 | 换成探测 MCP server 端口 9381 是否在线 |
| 3. 生产者-消费者队列 | `asyncio.Queue`、多消费者 | 换成法条切分入库流水线 |

### 交互口令

| 你说 | 我做 |
|------|------|
| `继续` | 进入下一阶段 |
| `跳到阶段 X` | 直接推进到该阶段 |
| `生成代码` | 给出当前阶段完整可运行代码 |
| `出题` | 给当前阶段的练习 |
| `审查` | 审你的代码并给改进建议 |
| `排错` | 按排错流程走：读错误 → 最小复现 → 二分定位 → 查官方文档 → 搜 issue → 看源码 → 提问 |
| `总结` | 输出当前阶段笔记 |
| `项目` | 进入项目驱动阶段 |

### 排错流程（阶段 8 会展开）

```
读错误信息 → 最小复现 → 二分定位 → 查官方文档 → 搜 issue → 看源码 → 提问
```

多数 asyncio 问题的答案在第一步和第二步就已经出现了。
